# Cycle 3 - Inference

## Research Question

**Is the proportion of students who felt sad or hopeless different between male and female students?**

This notebook focuses only on the formal statistical inference. Descriptive EDA was completed in the previous notebook.

## 1. Hypotheses

Let:

- \(p_{female}\) = the population proportion of female students who felt sad or hopeless.
- \(p_{male}\) = the population proportion of male students who felt sad or hopeless.

The hypotheses are:

\[
H_0: p_{female} = p_{male}
\]

\[
H_1: p_{female} \ne p_{male}
\]

This is a **two-sided test** because the research question asks whether the two proportions are different.

## 2. Method

Because `SadOrHopeless` is a binary response variable, the appropriate method is a **two-proportion z-test**.

The estimated difference is:

\[
\hat{p}_{female} - \hat{p}_{male}
\]

The significance level is:

\[
\alpha = 0.05
\]

In [65]:
# ============================================
# Cycle 3 - Two-Proportion Inference
# ============================================

import pandas as pd
import numpy as np
from scipy.stats import norm
from pathlib import Path

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", "{:.4f}".format)

# Paths
processed_path = Path("../data/processed/cycle3_selected_variables.csv")
table_dir = Path("../outputs/tables")
summary_dir = Path("../outputs/summary")

table_dir.mkdir(parents=True, exist_ok=True)
summary_dir.mkdir(parents=True, exist_ok=True)

# Load cleaned data
# Coding:
# WhatIsYourSex: 1 = Female, 0 = Male
# SadOrHopeless: 1 = Yes, 0 = No

df_analysis = pd.read_csv(processed_path)

print("Cleaned data shape:", df_analysis.shape)
display(df_analysis.head())

Cleaned data shape: (13833, 2)


,WhatIsYourSex,SadOrHopeless
0,0,1
1,0,0
2,0,1
3,1,1
4,1,1


## 3. Group Summary

This table gives the sample size, number of `Yes` responses, number of `No` responses, and sample proportion for each gender group.

In [66]:
# ============================================
# Group Summary
# ============================================

group_summary = df_analysis.groupby("WhatIsYourSex").agg(
    sample_size=("SadOrHopeless", "count"),
    yes_count=("SadOrHopeless", "sum"),
    sample_proportion=("SadOrHopeless", "mean")
).reset_index()

group_summary["sex_group"] = group_summary["WhatIsYourSex"].map({
    1: "Female",
    0: "Male"
})

group_summary["no_count"] = group_summary["sample_size"] - group_summary["yes_count"]
group_summary["percentage"] = group_summary["sample_proportion"] * 100

group_summary = group_summary[
    ["sex_group", "sample_size", "yes_count", "no_count", "sample_proportion", "percentage"]
]

# Save group summary
group_summary.to_csv(
    table_dir / "cycle3_inference_group_summary.csv",
    index=False
)

display(group_summary)

,sex_group,sample_size,yes_count,no_count,sample_proportion,percentage
0,Male,6893,1567,5326,0.2273,22.7332
1,Female,6940,2580,4360,0.3718,37.1758


## 4. Inference Calculation

The test compares the difference in sample proportions:

\[
\hat{p}_{female} - \hat{p}_{male}
\]

For the hypothesis test, the pooled proportion is used under \(H_0\). For the confidence interval, the unpooled standard error is used.

In [67]:
# ============================================
# Two-Proportion Z-Test and 95% Confidence Interval
# ============================================

alpha = 0.05
confidence_level = 0.95

# Extract group statistics
female = group_summary[group_summary["sex_group"] == "Female"].iloc[0]
male = group_summary[group_summary["sex_group"] == "Male"].iloc[0]

x_female = female["yes_count"]
n_female = female["sample_size"]
p_female = female["sample_proportion"]

x_male = male["yes_count"]
n_male = male["sample_size"]
p_male = male["sample_proportion"]

# Estimated difference
diff = p_female - p_male

# Hypothesis test: pooled standard error
p_pool = (x_female + x_male) / (n_female + n_male)
se_test = np.sqrt(
    p_pool * (1 - p_pool) * (1 / n_female + 1 / n_male)
)
z_stat = diff / se_test
p_value = 2 * (1 - norm.cdf(abs(z_stat)))

# Confidence interval: unpooled standard error
z_critical = norm.ppf(1 - (1 - confidence_level) / 2)
se_ci = np.sqrt(
    (p_female * (1 - p_female) / n_female) +
    (p_male * (1 - p_male) / n_male)
)
ci_lower = diff - z_critical * se_ci
ci_upper = diff + z_critical * se_ci

# Decision
if p_value < alpha:
    decision = "Reject H0"
else:
    decision = "Do not reject H0"

if diff > 0:
    higher_group = "Female"
elif diff < 0:
    higher_group = "Male"
else:
    higher_group = "Neither group"

print("Inference calculation completed.")

Inference calculation completed.


## 5. Main Inference Result Table

This table is the main statistical result. It is organized so that the key statistics can be read at a glance.

In [68]:
# ============================================
# Main Inference Result Table
# ============================================

main_result = pd.DataFrame({
    "Statistic": [
        "Method",
        "Significance level alpha",
        "H0",
        "H1",
        "Female sample size",
        "Female yes count",
        "Female no count",
        "Female sample proportion",
        "Male sample size",
        "Male yes count",
        "Male no count",
        "Male sample proportion",
        "Estimated difference: p_female - p_male",
        "95% CI lower bound",
        "95% CI upper bound",
        "Pooled proportion under H0",
        "SE for z-test",
        "z statistic",
        "p-value",
        "Decision",
        "Group with higher sample proportion"
    ],
    "Value": [
        "Two-proportion z-test",
        f"{alpha:.2f}",
        "p_female = p_male",
        "p_female != p_male",
        f"{int(n_female)}",
        f"{int(x_female)}",
        f"{int(n_female - x_female)}",
        f"{p_female:.4f} ({p_female * 100:.2f}%)",
        f"{int(n_male)}",
        f"{int(x_male)}",
        f"{int(n_male - x_male)}",
        f"{p_male:.4f} ({p_male * 100:.2f}%)",
        f"{diff:.4f} ({diff * 100:.2f} percentage points)",
        f"{ci_lower:.4f}",
        f"{ci_upper:.4f}",
        f"{p_pool:.4f}",
        f"{se_test:.4f}",
        f"{z_stat:.4f}",
        f"{p_value:.4f}",
        decision,
        higher_group
    ]
})

# Save readable result table
main_result.to_csv(
    table_dir / "cycle3_main_inference_result.csv",
    index=False
)

display(main_result)

,Statistic,Value
0,Method,Two-proportion z-test
1,Significance level alpha,0.05
2,H0,p_female = p_male
3,H1,p_female != p_male
4,Female sample size,6940
5,Female yes count,2580
6,Female no count,4360
7,Female sample proportion,0.3718 (37.18%)
8,Male sample size,6893
9,Male yes count,1567


## 6. Compact Result Table

This compact table is useful for the report, README, or one-slide summary.

In [69]:
# ============================================
# Compact Result Table
# ============================================

compact_result = pd.DataFrame({
    "Female proportion": [p_female],
    "Male proportion": [p_male],
    "Difference (Female - Male)": [diff],
    "95% CI": [f"[{ci_lower:.4f}, {ci_upper:.4f}]"],
    "z statistic": [z_stat],
    "p-value": [p_value],
    "Decision at alpha = 0.05": [decision]
})

compact_result.to_csv(
    table_dir / "cycle3_compact_inference_result.csv",
    index=False
)

display(compact_result)

,Female proportion,Male proportion,Difference (Female - Male),95% CI,z statistic,p-value,Decision at alpha = 0.05
0,0.3718,0.2273,0.1444,"[0.1294, 0.1595]",18.5374,0.0000,Reject H0


## 7. Assumption Check

For a two-proportion z-test, each group should have a large enough number of `Yes` and `No` responses. This notebook checks whether both counts are at least 10 in each group.

In [70]:
# ============================================
# Assumption Check
# ============================================

assumption_check = pd.DataFrame({
    "Group": ["Female", "Male"],
    "Yes count": [x_female, x_male],
    "No count": [n_female - x_female, n_male - x_male],
    "Yes >= 10": [x_female >= 10, x_male >= 10],
    "No >= 10": [(n_female - x_female) >= 10, (n_male - x_male) >= 10]
})

assumption_check.to_csv(
    table_dir / "cycle3_assumption_check.csv",
    index=False
)

display(assumption_check)

,Group,Yes count,No count,Yes >= 10,No >= 10
0,Female,2580,4360,True,True
1,Male,1567,5326,True,True


## 8. Final Conclusion

The conclusion below uses the p-value, the estimated difference, and the 95% confidence interval. Since the dataset is observational survey data, the result should be interpreted as a difference in proportions, not as a causal effect.

In [71]:
# ============================================
# Final Conclusion
# ============================================
if p_value < alpha:
    conclusion = (
        f"At alpha = {alpha}, the p-value is {p_value:.4f}, which is less than {alpha}.\n\n"
        f"Therefore, we reject H0.\n\n"
        f"There is statistically significant evidence that the proportion of students who felt sad or hopeless "
        f"is different between female and male students.\n\n"
        f"The female sample proportion was {p_female:.4f} ({p_female * 100:.2f}%).\n"
        f"The male sample proportion was {p_male:.4f} ({p_male * 100:.2f}%).\n\n"
        f"The estimated difference p_female - p_male was {diff:.4f} "
        f"({diff * 100:.2f} percentage points).\n\n"
        f"The 95% confidence interval for the difference was from {ci_lower:.4f} to {ci_upper:.4f}.\n\n"
        f"The {higher_group.lower()} group had the higher sample proportion."
    )

else:
    conclusion = (
        f"At alpha = {alpha}, the p-value is {p_value:.4f}, which is greater than {alpha}.\n\n"
        f"Therefore, we do not reject H0.\n\n"
        f"There is not enough statistically significant evidence to conclude that the proportion of students "
        f"who felt sad or hopeless is different between female and male students.\n\n"
        f"The female sample proportion was {p_female:.4f} ({p_female * 100:.2f}%).\n"
        f"The male sample proportion was {p_male:.4f} ({p_male * 100:.2f}%).\n\n"
        f"The estimated difference p_female - p_male was {diff:.4f} "
        f"({diff * 100:.2f} percentage points).\n\n"
        f"The 95% confidence interval for the difference was from {ci_lower:.4f} to {ci_upper:.4f}."
    )
with open(summary_dir / "cycle3_final_interpretation.txt", "w", encoding="utf-8") as f:
    f.write(conclusion)

print(conclusion)

At alpha = 0.05, the p-value is 0.0000, which is less than 0.05.

Therefore, we reject H0.

There is statistically significant evidence that the proportion of students who felt sad or hopeless is different between female and male students.

The female sample proportion was 0.3718 (37.18%).
The male sample proportion was 0.2273 (22.73%).

The estimated difference p_female - p_male was 0.1444 (14.44 percentage points).

The 95% confidence interval for the difference was from 0.1294 to 0.1595.

The female group had the higher sample proportion.


## Output Files

This notebook saves the following files:

| File | Location | Purpose |
|---|---|---|
| `cycle3_inference_group_summary.csv` | `outputs/tables/` | Group-level summary |
| `cycle3_main_inference_result.csv` | `outputs/tables/` | Full readable inference result |
| `cycle3_compact_inference_result.csv` | `outputs/tables/` | Compact result for report or slide |
| `cycle3_assumption_check.csv` | `outputs/tables/` | Large-sample condition check |
| `cycle3_final_interpretation.txt` | `outputs/summary/` | Final written conclusion |